# Notebook 03: Model Evaluation, Explainability & Maintenance Risk Intelligence

**Project**: Industrial Predictive Maintenance Platform & Failure Prevention  
**Track**: Advanced Data Science & AI (Project 5)  
**Author / Responsible Teammate**: Member 3  
**Dataset**: UCI AI4I 2020 Predictive Maintenance Dataset  

---

## Executive Summary & Objectives
1. **Evaluation Framework**: Compute PR-AUC, ROC-AUC, Precision, Recall, F1-Score, and False Negative Rate (FNR) on unseen test data.
2. **Threshold Optimization**: Formulate business cost function ($C_{total} = C_{FN} \cdot FN + C_{FP} \cdot FP$) with a $10:1$ cost ratio ($C_{FN}=\$5,000$, $C_{FP}=\$500$) to minimize financial loss from missed machine failures.
3. **Error Analysis**: Diagnostic investigation into False Positives (FP) and False Negatives (FN).
4. **Explainability**: Extract Gini importance, Permutation Feature Importance, Global SHAP summary, and Local SHAP Waterfall prediction breakdowns.
5. **Maintenance Risk Intelligence Tiers**: Categorize machine operational status into 4 actionable risk tiers (**Low**, **Medium**, **High**, **Critical**) and failure mode heuristic rules.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance
import shap

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__'))) if '__file__' in locals() else '..'
DATA_PATH = os.path.join(REPO_DIR, 'data', 'ai4i2020_feature_ready.csv')
MODEL_PATH = os.path.join(REPO_DIR, 'models', 'gradient_boosting_final_model.pkl')
SCALER_PATH = os.path.join(REPO_DIR, 'models', 'feature_scaler.pkl')
FIGURES_DIR = os.path.join(REPO_DIR, 'figures')
MODELS_DIR = os.path.join(REPO_DIR, 'models')
DOCS_DIR = os.path.join(REPO_DIR, 'docs')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print('Environment and libraries imported successfully!')

## Section 1: Data & Pre-Trained Artifact Loading
We load the feature-engineered dataset (`ai4i2020_feature_ready.csv`) and apply the exact 70/15/15 stratified train/validation/test split (`random_state=42`) established by Member 2 to ensure absolute reproducibility.

In [ ]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=['machine_failure'])
y = df['machine_failure']

RANDOM_STATE = 42
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

numeric_cols = [
    'air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm',
    'torque_nm', 'tool_wear_min', 'temp_diff_k', 'power_w',
    'tool_wear_torque_product', 'speed_torque_ratio'
]

scaler = joblib.load(SCALER_PATH)
model = joblib.load(MODEL_PATH)

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f'Train: {X_train.shape[0]} rows | Val: {X_val.shape[0]} rows | Test: {X_test.shape[0]} rows')

## Section 2: Comprehensive Model Evaluation & Curves
We calculate model metrics (Precision, Recall, F1, PR-AUC, ROC-AUC, FNR) on the test set under the default threshold ($t = 0.50$).

In [ ]:
y_val_proba = model.predict_proba(X_val_scaled)[:, 1]
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

y_val_pred_def = (y_val_proba >= 0.5).astype(int)
y_test_pred_def = (y_test_proba >= 0.5).astype(int)

def get_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    pr_auc = average_precision_score(y_true, y_proba)
    roc_auc = roc_auc_score(y_true, y_proba)
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return {
        'precision': float(prec), 'recall': float(rec), 'f1': float(f1),
        'pr_auc': float(pr_auc), 'roc_auc': float(roc_auc), 'fnr': float(fnr),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
    }

test_metrics_def = get_metrics(y_test, y_test_pred_def, y_test_proba)
print('Default Threshold (t=0.50) Test Metrics:')
for k, v in test_metrics_def.items():
    print(f'  {k}: {v}')

# Plot ROC and PR Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[0].plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'Gradient Boosting (ROC-AUC = {test_metrics_def["roc_auc"]:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_title('Receiver Operating Characteristic (ROC) Curve', fontweight='bold')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].legend(loc='lower right')

prec_c, rec_c, _ = precision_recall_curve(y_test, y_test_proba)
axes[1].plot(rec_c, prec_c, color='#2ca02c', lw=2.5, label=f'Gradient Boosting (PR-AUC = {test_metrics_def["pr_auc"]:.4f})')
axes[1].axhline(y=y_test.mean(), color='crimson', linestyle='--', label=f'Baseline Chance ({y_test.mean():.4f})')
axes[1].set_title('Precision-Recall (PR) Curve', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '08_roc_pr_curves.png'), dpi=300)
plt.show()

## Section 3: Business Cost Simulation & Threshold Optimization
We set up financial cost parameters ($C_{FN} = \$5,000$, $C_{FP} = \$500$) and search for optimal threshold $t^*$ on the validation set to minimize financial loss.

In [ ]:
C_FN = 5000
C_FP = 500

thresholds = np.linspace(0.01, 0.99, 99)
val_costs = []
val_fnrs = []
val_f1s = []

for t in thresholds:
    val_pred = (y_val_proba >= t).astype(int)
    cm_v = confusion_matrix(y_val, val_pred)
    tn, fp, fn, tp = cm_v.ravel()
    val_costs.append(C_FN * fn + C_FP * fp)
    val_fnrs.append(fn / (fn + tp) if (fn + tp) > 0 else 0)
    val_f1s.append(f1_score(y_val, val_pred, zero_division=0))

best_idx = np.argmin(val_costs)
optimal_threshold = thresholds[best_idx]
print(f'Cost-Optimal Threshold found on Validation Set: t* = {optimal_threshold:.4f}')

y_test_pred_opt = (y_test_proba >= optimal_threshold).astype(int)
test_metrics_opt = get_metrics(y_test, y_test_pred_opt, y_test_proba)

cost_test_def = int(C_FN * test_metrics_def['fn'] + C_FP * test_metrics_def['fp'])
cost_test_opt = int(C_FN * test_metrics_opt['fn'] + C_FP * test_metrics_opt['fp'])
cost_test_reactive = int(C_FN * y_test.sum())

print(f'Test Financial Cost Comparison:')
print(f'  Reactive (No Model):   ${cost_test_reactive:,}')
print(f'  Default Threshold:     ${cost_test_def:,} (FN={test_metrics_def["fn"]}, FP={test_metrics_def["fp"]})')
print(f'  Optimal Threshold:     ${cost_test_opt:,} (FN={test_metrics_opt["fn"]}, FP={test_metrics_opt["fp"]})')
print(f'  Savings vs Default:    ${cost_test_def - cost_test_opt:,} ({(cost_test_def - cost_test_opt)/cost_test_def*100:.1f}%)')

# Plot Cost Minimization & Confusion Matrices
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.set_xlabel('Decision Threshold (t)', fontsize=12)
ax1.set_ylabel(f'Total Business Cost ($)', color='#d62728', fontsize=12)
ax1.plot(thresholds, val_costs, color='#d62728', lw=2.5, label='Validation Total Cost ($)')
ax1.axvline(x=0.50, color='gray', linestyle='--', label='Default (0.50)')
ax1.axvline(x=optimal_threshold, color='green', linestyle='-', lw=2, label=f'Optimal ({optimal_threshold:.2f})')

ax2 = ax1.twinx()
ax2.set_ylabel('F1 Score / FNR', color='#1f77b4', fontsize=12)
ax2.plot(thresholds, val_f1s, color='#1f77b4', linestyle='-.', label='F1 Score')
ax2.plot(thresholds, val_fnrs, color='#ff7f0e', linestyle=':', label='False Negative Rate')

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper center')
plt.title('Threshold Cost Optimization & Business Loss Minimization', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '10_threshold_cost_optimization.png'), dpi=300)
plt.show()

# Side-by-Side Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test, y_test_pred_def), annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['No Failure', 'Failure'], yticklabels=['No Failure', 'Failure'], annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title(f'Default Threshold (t = 0.50)\nRecall: {test_metrics_def["recall"]:.2%}, FNR: {test_metrics_def["fnr"]:.2%}', fontweight='bold')

sns.heatmap(confusion_matrix(y_test, y_test_pred_opt), annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False,
            xticklabels=['No Failure', 'Failure'], yticklabels=['No Failure', 'Failure'], annot_kws={'size': 14, 'weight': 'bold'})
axes[1].set_title(f'Cost-Optimal Threshold (t = {optimal_threshold:.2f})\nRecall: {test_metrics_opt["recall"]:.2%}, FNR: {test_metrics_opt["fnr"]:.2%}', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '09_confusion_matrices.png'), dpi=300)
plt.show()

## Section 4: Explainability & SHAP Feature Analysis
We inspect model feature importance (Gini & Permutation) and generate global & local SHAP explanations.

In [ ]:
gini_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)
perm_res = permutation_importance(model, X_val_scaled, y_val, n_repeats=5, random_state=RANDOM_STATE, scoring='f1')
perm_imp = pd.Series(perm_res.importances_mean, index=X.columns).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
gini_imp.plot(kind='barh', color='#1f77b4', ax=axes[0])
axes[0].set_title('Gini Feature Importance (Model Built-in)', fontweight='bold')
perm_imp.plot(kind='barh', color='#ff7f0e', ax=axes[1])
axes[1].set_title('Permutation Feature Importance (Validation F1)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '11_feature_importance.png'), dpi=300)
plt.show()

# SHAP Global & Local Analysis
explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(X_test_scaled, check_additivity=False)

fig, ax = plt.subplots(figsize=(10, 6))
mean_shap = pd.Series(np.abs(shap_vals).mean(axis=0), index=X.columns).sort_values(ascending=True)
mean_shap.plot(kind='barh', color='#d62728', ax=ax)
ax.set_title('SHAP Global Feature Impact (Mean Absolute SHAP Value)', fontweight='bold')
ax.set_xlabel('mean(|SHAP value|)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '12_shap_global_summary.png'), dpi=300)
plt.show()

# Local Explanation Waterfall Plot
tp_idx = np.where((y_test == 1) & (y_test_pred_def == 1))[0][0]
plt.figure(figsize=(9, 5))
sample_shap = pd.Series(shap_vals[tp_idx], index=X.columns).sort_values(ascending=True)
sample_shap.plot(kind='barh', color=np.where(sample_shap >= 0, '#d62728', '#1f77b4'))
plt.title(f'SHAP Local Feature Contributions — True Positive Machine Failure (Prob={y_test_proba[tp_idx]:.2%})', fontweight='bold')
plt.xlabel('SHAP Value (Contribution to Failure Risk)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '13_shap_local_waterfall.png'), dpi=300)
plt.show()

## Section 5: Maintenance Risk Tiers & Failure Classification
We categorize test machine instances into 4 operational risk tiers: **Low Risk**, **Medium Risk**, **High Risk**, and **Critical Risk**.

In [ ]:
t_low = 0.15
t_opt = float(optimal_threshold)
t_crit = 0.75

def assign_risk(prob):
    if prob < t_low:
        return 'Low Risk (Normal Operation)'
    elif prob < t_opt:
        return 'Medium Risk (Monitor Closely)'
    elif prob < t_crit:
        return 'High Risk (Schedule Maintenance)'
    else:
        return 'Critical Risk (Immediate Action Required)'

risk_tiers = pd.Series(y_test_proba).apply(assign_risk).value_counts()
print('Maintenance Risk Tier Distribution (Test Set):')
print(risk_tiers)

fig, ax = plt.subplots(figsize=(9, 5))
colors = {'Low Risk (Normal Operation)': '#2ca02c', 'Medium Risk (Monitor Closely)': '#ff7f0e',
          'High Risk (Schedule Maintenance)': '#e377c2', 'Critical Risk (Immediate Action Required)': '#d62728'}
tier_order = ['Low Risk (Normal Operation)', 'Medium Risk (Monitor Closely)', 'High Risk (Schedule Maintenance)', 'Critical Risk (Immediate Action Required)']
counts_ordered = [int(risk_tiers.get(t, 0)) for t in tier_order]

bars = ax.bar(tier_order, counts_ordered, color=[colors[t] for t in tier_order], edgecolor='black', alpha=0.85)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f'{int(yval)}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Machinery Distribution Across Maintenance Risk Tiers', fontweight='bold')
ax.set_ylabel('Number of Machines')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '14_maintenance_risk_tiers.png'), dpi=300)
plt.show()

## Section 6: Exporting Artifacts & Metrics JSONs
We export `models/evaluation_summary.json` and `models/risk_thresholds.json` to complete Member 3's hand-off for Member 5 & Member 6.

In [ ]:
def json_serialize(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

eval_summary = {
    'model_name': 'GradientBoostingClassifier',
    'dataset': 'AI4I 2020 Predictive Maintenance',
    'cost_parameters': {'cost_fn': int(C_FN), 'cost_fp': int(C_FP), 'cost_ratio': '10:1'},
    'default_threshold_0.50': test_metrics_def,
    'cost_optimal_threshold': {
        'optimal_threshold': float(optimal_threshold),
        **test_metrics_opt
    },
    'business_impact': {
        'cost_reactive_no_model_usd': int(cost_test_reactive),
        'savings_vs_default_usd': int(cost_test_def - cost_test_opt),
        'savings_vs_default_pct': float(round((cost_test_def - cost_test_opt)/cost_test_def*100, 2)) if cost_test_def > 0 else 0.0,
        'savings_vs_reactive_usd': int(cost_test_reactive - cost_test_opt),
        'savings_vs_reactive_pct': float(round((cost_test_reactive - cost_test_opt)/cost_test_reactive*100, 2))
    }
}

with open(os.path.join(MODELS_DIR, 'evaluation_summary.json'), 'w') as f:
    json.dump(eval_summary, f, indent=4, default=json_serialize)

risk_config = {
    'optimal_threshold': float(optimal_threshold),
    'threshold_low': float(t_low),
    'threshold_medium': float(optimal_threshold),
    'threshold_critical': float(t_crit),
    'risk_tiers': {
        'Low': f'P < {t_low}',
        'Medium': f'{t_low} <= P < {optimal_threshold:.2f}',
        'High': f'{optimal_threshold:.2f} <= P < {t_crit}',
        'Critical': f'P >= {t_crit}'
    },
    'cost_fn_usd': int(C_FN),
    'cost_fp_usd': int(C_FP)
}

with open(os.path.join(MODELS_DIR, 'risk_thresholds.json'), 'w') as f:
    json.dump(risk_config, f, indent=4, default=json_serialize)

print('All evaluation artifacts and config JSON files successfully exported!')